In [5]:
import numpy as np
import random

# ==========================================
# Grid World
# ==========================================

rows = 4
cols = 4

start_state = (0, 0)
goal_state = (3, 3)

# Obstacles
obstacles = [(1, 1), (2, 1)]

# Actions
# 0 = Up
# 1 = Down
# 2 = Left
# 3 = Right

actions = 4

# ==========================================
# State Mapping
# ==========================================

def state_number(position):
    return position[0] * cols + position[1]

num_states = rows * cols

Q = np.zeros((num_states, actions))

# ==========================================
# Hyperparameters
# ==========================================

learning_rate = 0.8
discount_factor = 0.9

epsilon = 1.0
epsilon_decay = 0.995
epsilon_min = 0.01

episodes = 1000

# ==========================================
# Environment Function
# ==========================================

def step(state, action):

    r, c = state

    if action == 0:      # Up
        r -= 1

    elif action == 1:    # Down
        r += 1

    elif action == 2:    # Left
        c -= 1

    elif action == 3:    # Right
        c += 1

    # Boundary check
    r = max(0, min(rows - 1, r))
    c = max(0, min(cols - 1, c))

    next_state = (r, c)

    # Obstacle
    if next_state in obstacles:
        return state, -100, True

    # Goal
    if next_state == goal_state:
        return next_state, 100, True

    # Normal move
    return next_state, -1, False

# ==========================================
# Training
# ==========================================

rewards = []

for episode in range(episodes):

    state = start_state
    total_reward = 0
    done = False

    while not done:

        state_idx = state_number(state)

        # Epsilon-Greedy
        if random.uniform(0, 1) < epsilon:
            action = random.randint(0, 3)
        else:
            action = np.argmax(Q[state_idx])

        next_state, reward, done = step(state, action)

        next_idx = state_number(next_state)

        # Q-Learning Update
        Q[state_idx, action] = Q[state_idx, action] + learning_rate * (
            reward +
            discount_factor * np.max(Q[next_idx]) -
            Q[state_idx, action]
        )

        state = next_state
        total_reward += reward

    rewards.append(total_reward)

    if epsilon > epsilon_min:
        epsilon *= epsilon_decay

# ==========================================
# Print Q-Table
# ==========================================

print("Q-Table\n")
print(np.round(Q, 2))

# ==========================================
# Learned Path
# ==========================================

print("\nOptimal Path")

state = start_state
path = [state]

while state != goal_state:

    action = np.argmax(Q[state_number(state)])

    next_state, reward, done = step(state, action)

    if next_state == state:
        break

    path.append(next_state)
    state = next_state

    if len(path) > 20:
        break

print(path)

# ==========================================
# Display Grid
# ==========================================

grid = np.full((rows, cols), ".")

for obs in obstacles:
    grid[obs] = "X"

grid[start_state] = "S"
grid[goal_state] = "G"

print("\nGrid")
for row in grid:
    print(" ".join(row))

Q-Table

[[ 48.46  54.95  48.46  54.95]
 [ 54.95 -44.05  48.46  62.17]
 [ 62.17  70.19  54.95  70.19]
 [ 70.19  79.1   62.14  70.19]
 [ 48.46  62.17  54.95 -44.13]
 [  0.     0.     0.     0.  ]
 [ 62.17  79.1  -28.81  79.1 ]
 [ 70.19  89.    70.19  79.1 ]
 [ 54.95  70.19  62.17 -36.83]
 [  0.     0.     0.     0.  ]
 [ 70.08  89.   -19.9   88.99]
 [ 79.1  100.    79.1   89.  ]
 [ 62.17  70.19  70.19  79.1 ]
 [-19.9   79.1   70.19  89.  ]
 [ 79.1   89.    79.1  100.  ]
 [  0.     0.     0.     0.  ]]

Optimal Path
[(0, 0), (1, 0), (2, 0), (3, 0), (3, 1), (3, 2), (3, 3)]

Grid
S . . .
. X . .
. X . .
. . . G


In [3]:
pip install pandas nltk scikit-learn transformers torch sentence-transformers scipy

  Using cached transformers-5.12.1-py3-none-any.whl.metadata (33 kB)
  Using cached sentence_transformers-5.6.0-py3-none-any.whl.metadata (18 kB)
  Using cached huggingface_hub-1.21.0-py3-none-any.whl.metadata (14 kB)
  Using cached regex-2026.6.28-cp313-cp313-win_amd64.whl.metadata (41 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached safetensors-0.8.0-cp310-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached click-8.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached hf_xet-1.5.1-cp37-abi3-win_amd64.whl.metadata (4.9 kB)
Using cached transformers-5.12.1-py3-none-any.whl (11.2 MB)
Using cached huggingface_hub-1.21.0-py3-none-any.whl (721 kB)
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   -- ------------------------------------- 0.3/4.0 MB ? eta -:--:--
   ------------- -------------------------- 1.3/4.0 MB 5.5 MB/s eta 0:00:01
   -------------------- ------------------- 2.1/4.0 MB 4.6 MB/s eta 0:00:01
   ----------

  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.51.0 requires protobuf<7,>=3.20, but you have protobuf 7.35.1 which is incompatible.


In [4]:
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.linear_model import LogisticRegression
from scipy.sparse import hstack

from sentence_transformers import SentenceTransformer

# --------------------------------------------------
# Download NLTK Resources
# --------------------------------------------------

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

# --------------------------------------------------
# Load Dataset
# --------------------------------------------------

df = pd.read_csv("emotion-labels-test.csv")

print(df.head())

text = df["text"].astype(str)
labels = df["label"]

# --------------------------------------------------
# Label Encoding
# --------------------------------------------------

encoder = LabelEncoder()

y = encoder.fit_transform(labels)

# --------------------------------------------------
# NLP Preprocessing
# --------------------------------------------------

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess(sentence):

    sentence = sentence.lower()

    sentence = re.sub(r"http\S+", "", sentence)
    sentence = re.sub(r"@\w+", "", sentence)
    sentence = re.sub(r"[^a-zA-Z ]", " ", sentence)

    tokens = word_tokenize(sentence)

    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
        if word not in stop_words and len(word) > 2
    ]

    return " ".join(tokens)

clean_text = text.apply(preprocess)

# --------------------------------------------------
# TF-IDF Features
# --------------------------------------------------

tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1,2),
    sublinear_tf=True
)

tfidf_features = tfidf.fit_transform(clean_text)

print("TF-IDF Shape:", tfidf_features.shape)

# --------------------------------------------------
# Transfer Learning (BERT)
# --------------------------------------------------

bert = SentenceTransformer("all-MiniLM-L6-v2")

bert_embeddings = bert.encode(
    clean_text.tolist(),
    show_progress_bar=True
)

print("BERT Shape:", bert_embeddings.shape)

# --------------------------------------------------
# Combine TF-IDF + BERT
# --------------------------------------------------

X = hstack([tfidf_features, bert_embeddings])

print("Combined Shape:", X.shape)

# --------------------------------------------------
# Train Test Split
# --------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# --------------------------------------------------
# Train Classifier
# --------------------------------------------------

model = LogisticRegression(
    max_iter=5000
)

model.fit(X_train, y_train)

# --------------------------------------------------
# Prediction
# --------------------------------------------------

pred = model.predict(X_test)

# --------------------------------------------------
# Evaluation
# --------------------------------------------------

print()

print("Accuracy")

print(accuracy_score(y_test, pred))

print()

print(classification_report(
    y_test,
    pred,
    target_names=encoder.classes_
))

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\SruthinJayakumaran\AppData\Roaming\nltk_data.
[nltk_data]     ..
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\SruthinJayakumaran\AppData\Roaming\nltk_data.
[nltk_data]     ..
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\SruthinJayakumaran\AppData\Roaming\nltk_data.
[nltk_data]     ..
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\SruthinJayakumaran\AppData\Roaming\nltk_data.
[nltk_data]     ..
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\SruthinJayakumaran\AppData\Roaming\nltk_data.
[nltk_data]     ..
[nltk_data]   Package omw-1.4 is already up-to-date!


                                                text label
0  You must be knowing #blithe means (adj.)  Happ...   joy
1  Old saying 'A #smile shared is one gained for ...   joy
2  Bridget Jones' Baby was bloody hilarious 😅 #Br...   joy
3  @Elaminova sparkling water makes your life spa...   joy
4  I'm tired of everybody telling me to chill out...   joy
TF-IDF Shape: (3142, 10000)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\SruthinJayakumaran\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\SruthinJayakumaran\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/99 [00:00<?, ?it/s]

BERT Shape: (3142, 384)
Combined Shape: (3142, 10384)

Accuracy
0.7821939586645469

              precision    recall  f1-score   support

       anger       0.81      0.76      0.78       152
        fear       0.76      0.82      0.79       199
         joy       0.82      0.84      0.83       143
     sadness       0.76      0.69      0.72       135

    accuracy                           0.78       629
   macro avg       0.78      0.78      0.78       629
weighted avg       0.78      0.78      0.78       629

